In [1]:
!pip -q uninstall -y pyspark spark-nlp sparknlp dataproc-spark-connect
!pip -q install pyspark==3.4.4 spark-nlp==6.3.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.2/746.2 kB 50.3 MB/s eta 0:00:00


In [2]:
import sys
import pandas as pd
import sparknlp

from pyspark.ml import Pipeline
from pyspark.sql.types import StringType
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import (
    Tokenizer,
    BertForSequenceClassification,
    WordEmbeddingsModel,
    NerDLModel,
    NerConverter,
    UniversalSentenceEncoder,
    ClassifierDLModel
)

spark = sparknlp.start()

print("Python:", sys.version.split()[0])
print("Spark NLP:", sparknlp.version())
print("Apache Spark:", spark.version)

Python: 3.11.13
Spark NLP: 6.3.2
Apache Spark: 3.4.4


**Detect Sentiment & Emotion**

In [3]:
emotion_texts = [
    "The new bike lanes on Main Street are fantastic! I feel so much safer cycling to work now.",
    "I've been waiting 3 months for the pothole on Oak Avenue to be fixed. Someone is going to get hurt!",
    "My kids love the new playground equipment at Lincoln Park. Thank you for listening to the community!",
    "The construction noise at 6 AM every morning is making me lose sleep. This has been going on for weeks.",
    "Why is the city spending millions on a stadium when our schools need repairs? This makes me angry.",
    "I'm worried about the lack of street lighting near the high school. It's not safe for students walking home.",
    "The new 311 app is so confusing. I tried for an hour to submit a request and couldn't figure it out.",
    "Thank you to the snow removal crew! They did an amazing job clearing our street quickly."
]

print("Number of comments:", len(emotion_texts))

Number of comments: 8


In [4]:
emotion_document = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

emotion_tokenizer = (
    Tokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
)

emotion_model = (
    BertForSequenceClassification
    .pretrained("bert_sequence_classifier_emotion", "en")
    .setInputCols(["token", "document"])
    .setOutputCol("prediction")
)

emotion_pipeline = Pipeline(
    stages=[
        emotion_document,
        emotion_tokenizer,
        emotion_model
    ]
)

emotion_input = (
    spark.createDataFrame(emotion_texts, StringType())
    .toDF("text")
)

emotion_result = (
    emotion_pipeline
    .fit(emotion_input)
    .transform(emotion_input)
)

emotion_rows = emotion_result.select(
    "text", "prediction"
).collect()

emotion_records = []

for number, row in enumerate(emotion_rows, start=1):
    prediction = row.prediction[0]
    label = prediction.result
    score = prediction.metadata.get(label)

    emotion_records.append({
        "comment_id": f"E{number:02d}",
        "text": row.text,
        "emotion": label,
        "model_score": float(score) if score is not None else None
    })

emotion_table = pd.DataFrame(emotion_records)

display(emotion_table)

bert_sequence_classifier_emotion download started this may take some time.
Approximate size to download 391.1 MB
[OK!]


,comment_id,text,emotion,model_score
0,E01,The new bike lanes on Main Street are fantasti...,joy,0.996363
1,E02,I've been waiting 3 months for the pothole on ...,sadness,0.968860
2,E03,My kids love the new playground equipment at L...,joy,0.949659
3,E04,The construction noise at 6 AM every morning i...,anger,0.624587
4,E05,Why is the city spending millions on a stadium...,anger,0.993453
5,E06,I'm worried about the lack of street lighting ...,fear,0.785651
6,E07,The new 311 app is so confusing. I tried for a...,fear,0.676174
7,E08,Thank you to the snow removal crew! They did a...,joy,0.977450


**Recognize Entities**

In [5]:
infrastructure_texts = [
    "There's a broken streetlight at the corner of 5th Avenue and Elm Street. It's been out since January 15th.",
    "The sidewalk on Washington Boulevard between Park Street and City Hall is cracked and dangerous.",
    "Water main break on Grove Avenue near Roosevelt Elementary School. Reported at 8:00 AM today.",
    "The traffic signal at Main Street and Oak Avenue has been stuck on red since Tuesday morning.",
    "Graffiti on the Madison Park Community Center needs removal. It appeared over the weekend.",
    "Tree branches over Central Avenue near house number 482 are blocking street signs.",
    "The bus shelter at Maple Street and Route 42 has a leaking roof. Passengers are getting wet.",
    "Potholes on Lincoln Highway between mile markers 15 and 18 need urgent repair."
]

print("Number of reports:", len(infrastructure_texts))

Number of reports: 8


In [6]:
ner_document = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

ner_tokenizer = (
    Tokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
)

ner_embeddings = (
    WordEmbeddingsModel
    .pretrained("glove_100d")
    .setInputCols(["document", "token"])
    .setOutputCol("embeddings")
)

ner_model = (
    NerDLModel
    .pretrained("onto_100", "en")
    .setInputCols(["document", "token", "embeddings"])
    .setOutputCol("ner")
)

ner_converter = (
    NerConverter()
    .setInputCols(["document", "token", "ner"])
    .setOutputCol("ner_chunk")
)

ner_pipeline = Pipeline(
    stages=[
        ner_document,
        ner_tokenizer,
        ner_embeddings,
        ner_model,
        ner_converter
    ]
)

ner_input = (
    spark.createDataFrame(infrastructure_texts, StringType())
    .toDF("text")
)

ner_result = (
    ner_pipeline
    .fit(ner_input)
    .transform(ner_input)
)

ner_rows = ner_result.select("text", "ner_chunk").collect()

entity_records = []

for number, row in enumerate(ner_rows, start=1):
    for entity in row.ner_chunk:
        entity_records.append({
            "report_id": f"N{number:02d}",
            "original_text": row.text,
            "extracted_text": entity.result,
            "entity_type": entity.metadata.get("entity")
        })

entity_table = pd.DataFrame(
    entity_records,
    columns=[
        "report_id",
        "original_text",
        "extracted_text",
        "entity_type"
    ]
)

display(entity_table)

print("Reports processed:", len(ner_rows))
print("Reports with no extracted entities:", [
    f"N{i:02d}"
    for i, row in enumerate(ner_rows, start=1)
    if not row.ner_chunk
])

glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
onto_100 download started this may take some time.
Approximate size to download 13.5 MB
[OK!]


,report_id,original_text,extracted_text,entity_type
0,N01,There's a broken streetlight at the corner of ...,5th Avenue,FAC
1,N01,There's a broken streetlight at the corner of ...,Elm Street,FAC
2,N01,There's a broken streetlight at the corner of ...,January 15th,DATE
3,N02,The sidewalk on Washington Boulevard between P...,Washington Boulevard,FAC
4,N02,The sidewalk on Washington Boulevard between P...,Park Street and City Hall,FAC
5,N03,Water main break on Grove Avenue near Roosevel...,Grove Avenue,FAC
6,N03,Water main break on Grove Avenue near Roosevel...,Roosevelt Elementary School,FAC
7,N03,Water main break on Grove Avenue near Roosevel...,8:00 AM,TIME
8,N03,Water main break on Grove Avenue near Roosevel...,today,DATE
9,N04,The traffic signal at Main Street and Oak Aven...,Main Street,FAC


Reports processed: 8
Reports with no extracted entities: []


In [7]:
!pip install --upgrade -q spark-nlp-display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 8.3 MB/s eta 0:00:00


In [14]:
from sparknlp_display import NerVisualizer

NerVisualizer().display(
    result=ner_result.select("document", "ner_chunk").collect()[1],
    label_col="ner_chunk",
    document_col="document"
)

In [10]:
from sparknlp_display import NerVisualizer

visualizer = NerVisualizer()
visual_rows = ner_result.select("document", "ner_chunk").collect()

for number, row in enumerate(visual_rows, start=1):
    print(f"Report {number}")

    visualizer.display(
        result=row,
        label_col="ner_chunk",
        document_col="document"
    )

Report 1


Report 2


Report 3


Report 4


Report 5


Report 6


Report 7


Report 8


**Classify Documents**

In [15]:
service_texts = [
    "The bus stop shelter on Route 42 near Maple Street is damaged. The roof is leaking.",
    "Bulk trash pickup request - items have been on curb for a week.",
    "Request to add more benches in Lincoln Park. Existing ones are always occupied.",
    "Parking meter on 3rd Street is malfunctioning and won't accept payment.",
    "Noise complaint about River Street bar - loud music past midnight on weekdays.",
    "Crosswalk needed at School Road and Highland Avenue intersection. Children crossing unsafely.",
    "Public library Wi-Fi has been down for three days. Many rely on this access.",
    "Stray dog spotted multiple times in Jefferson neighborhood. Public safety concern.",
    "Street signs at the corner of Oak and Elm are completely faded and unreadable.",
    "Request for community meeting about proposed zoning changes in downtown district."
]

print("Number of requests:", len(service_texts))

Number of requests: 10


In [16]:
classification_document = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

sentence_encoder = (
    UniversalSentenceEncoder
    .pretrained(lang="en")
    .setInputCols(["document"])
    .setOutputCol("sentence_embeddings")
)

classification_model = (
    ClassifierDLModel
    .pretrained("classifierdl_use_trec50", "en")
    .setInputCols(["sentence_embeddings"])
    .setOutputCol("class_prediction")
)

classification_pipeline = Pipeline(
    stages=[
        classification_document,
        sentence_encoder,
        classification_model
    ]
)

classification_input = (
    spark.createDataFrame(service_texts, StringType())
    .toDF("text")
)

classification_result = (
    classification_pipeline
    .fit(classification_input)
    .transform(classification_input)
)

classification_rows = classification_result.select(
    "text", "class_prediction"
).collect()

classification_records = []

for number, row in enumerate(classification_rows, start=1):
    classification_records.append({
        "request_id": f"C{number:02d}",
        "text": row.text,
        "model_label": row.class_prediction[0].result.strip()
    })

classification_table = pd.DataFrame(classification_records)

display(classification_table)

tfhub_use download started this may take some time.
Approximate size to download 923.7 MB
[OK!]
classifierdl_use_trec50 download started this may take some time.
Approximate size to download 21.2 MB
[OK!]


,request_id,text,model_label
0,C01,The bus stop shelter on Route 42 near Maple St...,LOC_other
1,C02,Bulk trash pickup request - items have been on...,NUM_count
2,C03,Request to add more benches in Lincoln Park. E...,LOC_other
3,C04,Parking meter on 3rd Street is malfunctioning ...,ENTY_other
4,C05,Noise complaint about River Street bar - loud ...,DESC_desc
5,C06,Crosswalk needed at School Road and Highland A...,HUM_ind
6,C07,Public library Wi-Fi has been down for three d...,LOC_other
7,C08,Stray dog spotted multiple times in Jefferson ...,HUM_ind
8,C09,Street signs at the corner of Oak and Elm are ...,DESC_reason
9,C10,Request for community meeting about proposed z...,DESC_reason


In [17]:
manual_routing_table = pd.DataFrame({
    "request_id": [f"C{i:02d}" for i in range(1, 11)],
    "text": service_texts,
    "manual_department": [""] * 10,
    "reason": [""] * 10
})

display(manual_routing_table)

,request_id,text,manual_department,reason
0,C01,The bus stop shelter on Route 42 near Maple St...,,
1,C02,Bulk trash pickup request - items have been on...,,
2,C03,Request to add more benches in Lincoln Park. E...,,
3,C04,Parking meter on 3rd Street is malfunctioning ...,,
4,C05,Noise complaint about River Street bar - loud ...,,
5,C06,Crosswalk needed at School Road and Highland A...,,
6,C07,Public library Wi-Fi has been down for three d...,,
7,C08,Stray dog spotted multiple times in Jefferson ...,,
8,C09,Street signs at the corner of Oak and Elm are ...,,
9,C10,Request for community meeting about proposed z...,,
